# CS-320 (Artificial Neural Networks & Deep Learning) Project P03
## Densely Connected Convolutional Networks (DenseNet, Huang et al., CVPR 2017)

**Team Members & Individual Viva Ownership:**
- **Mohammad Abdul Rehman** (Team & Paper Lead): `_DenseLayer`, `_DenseBlock`, `DenseNet`, Hand Parameter Formulas, CIFAR-10+ Reproduction Benchmark. Ablations: Growth rate $k \in \{8, 12, 24\}$, Bottleneck on/off.
- **Muhammad Wajahat** (Data & Pipeline Lead): Custom Datasets & DataLoaders, Train-Split-Only Normalization, `_Transition` Layer, Classification Metrics. Ablations: Compression factor $\theta \in \{0.5, 1.0\}$, Input Resolution ($28, 64, 128$).
- **Aiman Aslam** (Experiments Lead): Training & Validation Engine (AMP, Checkpointing), Sanity Checks, LR Range Test, Clinical Calibration ($\ge 95\%$ Sensitivity), Error Analysis, Gradio Demonstration. Ablations: Dropout $p \in \{0.0, 0.2\}$, Threshold Calibration.

---

## Section 0: Google Colab Setup, Repository Cloning & Package Installation

In [ ]:
# 1. Mount Google Drive for persistent and resumable checkpointing
from google.colab import drive
drive.mount('/content/drive')

# 2. Clone repository (replace with team repository URL) or navigate to root
!git clone https://github.com/your-username/ANN-SEM-Project.git /content/ANN-SEM-Project || true
%cd /content/ANN-SEM-Project

# 3. Install required dependencies
!pip install -q -r requirements.txt

# 4. Verify GPU runtime and environment
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print("Automatic Mixed Precision (AMP): Supported (torch.autocast)")

## Section 1: Data Verification, Pipeline Checks & Zero-Leakage Stats
- Strict separation of statistics: Mean and standard deviation are computed **strictly** on the 45,000 CIFAR-10 training split and the 4,708 PneumoniaMNIST training split.
- Data augmentation for CIFAR-10+: 4-pixel random crop and horizontal reflection flip.

In [ ]:
from src.data.cifar import get_cifar10_dataloaders, plot_cifar_sample_grid
from src.data.pneumonia import get_pneumonia_dataloaders, plot_pneumonia_sample_grid
from PIL import Image

# 1. Load CIFAR-10 DataLoaders
cifar_train, cifar_val, cifar_test, cifar_meta = get_cifar10_dataloaders(batch_size=64, smoke=False)
plot_cifar_sample_grid(cifar_train, save_path="results/cifar_sample_grid.png")

# 2. Load PneumoniaMNIST DataLoaders (size=64)
pneumo_train, pneumo_val, pneumo_test, pneumo_meta = get_pneumonia_dataloaders(batch_size=64, size=64, smoke=False)
plot_pneumonia_sample_grid(pneumo_train, save_path="results/pneumonia_sample_grid.png")

# Display sample grids
display(Image.open("results/cifar_sample_grid.png"))
display(Image.open("results/pneumonia_sample_grid.png"))

## Section 2: Architecture Verification, Analytical Parameter Formulas & Shape Progression
- Hand formula verification: $\text{BN} = 2 \cdot C_{in}$, $\text{Conv3x3} = 9 \cdot C_{in} \cdot k$.
- Proving Block 1 Layer 1 ($C_{in}=16, k=12$) equals **exactly 1,760 parameters**.
- Verifying DenseNet-40 matches $\sim 1.02\text{M}$ and DenseNet-BC-100 matches $\sim 0.77\text{M}$.
- Verifying custom DenseNet-121 matches `torchvision.models.densenet121` parameter-for-parameter (7,978,856 params).

In [ ]:
from src.utils.param_count import (
    verify_block1_layer1_params,
    verify_densenet40_params,
    verify_densenet_bc_100_params,
    compare_with_torchvision_densenet121,
    print_torchinfo_summary
)
from src.utils.shape_checks import generate_densenet40_shape_table
from src.models.densenet import densenet40_cifar

# 1. Run exact analytical parameter proofs
verify_block1_layer1_params()
verify_densenet40_params()
verify_densenet_bc_100_params()
compare_with_torchvision_densenet121()

# 2. Generate and display layer-by-layer shape table for DenseNet-40
generate_densenet40_shape_table(save_path="results/shape_table.md")

# 3. Print torchinfo summary table
model = densenet40_cifar()
print_torchinfo_summary(model, input_size=(1, 3, 32, 32))

## Section 3: Optimization Sanity Checks
1. **Initial Loss Check:** Confirms loss $\approx \ln(C) = \ln(10) \approx 2.3026$ for an untrained network on CIFAR-10.
2. **Gradient Flow Check:** Confirms non-zero gradients across 100% of trainable weight tensors.
3. **Single-Batch Overfitting:** Overfits a single mini-batch within 60 steps to ensure optimization capacity.

In [ ]:
from src.train.sanity import run_sanity_checks

sample_x, sample_y = next(iter(cifar_train))
sanity_results = run_sanity_checks(model, sample_x, sample_y, num_classes=10)

## Section 4: Learning Rate Range Test
- Exponential learning rate sweep from $10^{-6}$ to $2.0$ over 100 steps.
- Identifies point of steepest descent to validate base learning rate choice.

In [ ]:
from src.train.lr_range_test import run_lr_range_test
from src.train.engine import get_device
import torch.nn as nn

device = get_device()
criterion = nn.CrossEntropyLoss()
lrs, losses, suggested_lr = run_lr_range_test(
    model=model.to(device),
    dataloader=cifar_train,
    criterion=criterion,
    device=device,
    min_lr=1e-6,
    max_lr=2.0,
    num_iter=100,
    save_path="results/lr_range_test.png"
)

display(Image.open("results/lr_range_test.png"))

## Section 5: CIFAR-10+ Reproduction Benchmark
Trains and compares:
1. **DenseNet-40 ($k=12$)** (Paper error: 5.24%)
2. **DenseNet-BC-100 ($k=12$)** (Paper error: 4.51%)
3. **ResNet-56** (Paper baseline error: ~6.97%)

*Note:* On Colab, each 300-epoch run requires approx 2-2.5 hours under AMP. Checkpointing saves progress to Google Drive.

In [ ]:
# Execute full CIFAR-10 reproduction script
# Set --epochs 300 for full run, or --epochs 50 for reduced schedule, or --smoke for sanity
!python -m experiments.reproduce_cifar --model densenet40 --epochs 300 --batch-size 64 --lr 0.1

# Display training curves
display(Image.open("results/cifar10_densenet40_300ep_curves.png"))

## Section 6: Medical Transfer Extension on PneumoniaMNIST (size=64)
Evaluates:
1. **Simple CNN** (5-layer baseline adapted from MedMNIST notebook)
2. **Custom DenseNet** (DenseNet-BC adapted for 1-channel 64x64)
3. **Fine-Tuned DenseNet-121** (Torchvision ImageNet pretrained)
- Class imbalance handled via class-weighted CrossEntropyLoss.

In [ ]:
!python -m experiments.run_pneumonia --model all --size 64 --epochs 50 --batch-size 64 --lr 0.001

## Section 7: Systematic Ablation Studies (One-Factor-At-A-Time)
Evaluates each factor from the fixed baseline (`configs/baseline.yaml`):
- Growth rate $k \in \{8, 12, 24\}$
- Bottleneck on vs off
- Compression $\theta \in \{0.5, 1.0\}$
- Input resolution ($28, 64, 128$)
- Dropout $p \in \{0.0, 0.2\}$

In [ ]:
!python -m experiments.run_ablations --factor all

import pandas as pd
df_ablations = pd.read_csv("results/ablations.csv")
display(df_ablations)

# Display ablation plots
for f in ["growth_rate", "bottleneck", "compression", "resolution", "dropout"]:
    p = f"results/ablation_{f}.png"
    if os.path.exists(p):
        display(Image.open(p))

## Section 8: Diagnostic Error Analysis & Misclassifications
- Detailed confusion matrix and per-class error breakdown.
- Visual inspection grid of misclassified test samples with model confidence.

In [ ]:
from src.eval.error_analysis import analyze_misclassifications, plot_misclassified_grid
from src.eval.metrics import plot_confusion_matrix

misclass_data = analyze_misclassifications(
    model=model.to(device),
    dataloader=cifar_test,
    device=device,
    class_names=cifar_meta["classes"],
    max_samples=16
)
plot_misclassified_grid(
    misclass_data,
    class_names=cifar_meta["classes"],
    mean=cifar_meta["mean"],
    std=cifar_meta["std"],
    save_path="results/misclassified_grid.png"
)
display(Image.open("results/misclassified_grid.png"))

## Section 9: Clinical Calibration & Threshold Tuning
- Selecting threshold on validation set for $\ge 95\%$ sensitivity.
- Reliability diagram and Expected Calibration Error (ECE).
- Post-hoc temperature scaling.

In [ ]:
# Display calibration artifacts generated during PneumoniaMNIST experiments
for f in ["reliability.png", "tradeoff.png", "cm.png"]:
    for fname in os.listdir("results"):
        if f in fname:
            display(Image.open(os.path.join("results", fname)))

## Section 10: Interactive Gradio Diagnostic Explorer Launch
- Launches interactive diagnostic explorer with `share=True` for a public Colab URL.
- Features: Dataset selector, Sensitivity slider, and Figure 5 Feature-Reuse Heatmap / Grad-CAM.

In [ ]:
from app.gradio_demo import build_app

demo_app = build_app()
demo_app.launch(share=True)